# Model metrics history

Shows how the filled posts models' scores change across training runs, read from each run's files in the
pipeline-resources `models/` folder: run counts, **legacy** and **cross-validated** scores (kept apart, because
they are measured differently), change against the previous run and against a rolling baseline, and charts.
Use the breach tables to choose the alert tolerances.

**Before running:** the parameters below are set, and `boto3` and `matplotlib` are available on the instance.
This notebook only reads from S3; it writes nothing back.

**Stop the notebook instance when you finish** - SageMaker bills while it is running.

## Parameters

- `MODELS_PATH` - S3 folder holding every model's runs: `<model>/<version>/<run_number>/`.
- `STORAGE_OPTIONS` - Polars S3 options; leave `None` unless the region isn't found.
- `MODEL_NAMES` - the models to report on.
- `BASELINE_RUNS` - how many previous runs (same version) the drift baseline's median covers.
- `STEP_TOLERANCE`, `DRIFT_TOLERANCE` - trial values: the relative change in the wrong direction, against
  the previous run and against the baseline, that counts as a breach. Tune them from the tables below.
- `KEY_METRICS` - `(name, level, metric, fold, higher_is_better)` for each tracked score, read from the pooled
  rows of `metrics.parquet`. `ABSOLUTE_METRICS` are tracked as absolute values; `FLAG_METRICS` are only
  listed when above zero.
- Column and label names - defined once here (defaults are the training output's).

In [ ]:
MODELS_PATH = "s3://<pipeline-resources-bucket>/models/"
STORAGE_OPTIONS = None  # e.g. {"aws_region": "eu-west-2"} if Polars can't find the region
MODEL_NAMES = [
    "care_home_model",
    "non_res_with_dormancy_model",
    "non_res_without_dormancy_model",
]
BASELINE_RUNS = 5
STEP_TOLERANCE = 0.10
DRIFT_TOLERANCE = 0.20
CHART_COLUMNS = 3

# Names are defined once, here. Edit them if the training output changes.
BUCKET, MODELS_PREFIX = MODELS_PATH.removeprefix("s3://").split("/", 1)
RUN_PATTERN = MODELS_PREFIX + r"([^/]+)/([^/]+)/(\d+)/"

MODEL, VERSION, RUN, TIMESTAMP, EVALUATION = "model", "version", "run_number", "timestamp", "evaluation"
LEVEL, GROUP, FOLD, METRIC, VALUE, NAME = "level", "group", "fold", "metric", "value", "name"
HIGHER_IS_BETTER = "higher_is_better"
POOLED, ALL_GROUPS, ALL_ROWS = "pooled", "all", "all_rows"
LEGACY, CROSS_VALIDATED = "legacy", "cross_validated"

R2, RMSE, WAPE = "r2", "rmse", "weighted_absolute_percentage_error"
WITHIN_TEN = "proportion_of_model_predictions_within_ten"
WITHIN_TWENTY_FIVE = "proportion_of_model_predictions_within_twenty_five"
METADATA_METRICS = [R2, RMSE, WITHIN_TEN, WITHIN_TWENTY_FIVE]

KEY_METRICS = [
    ("posts_r2", "row_level_posts", R2, POOLED, True),
    ("posts_wape", "row_level_posts", WAPE, POOLED, False),
    ("posts_abs_bias", "row_level_posts", "bias", POOLED, False),
    ("posts_rmse", "row_level_posts", RMSE, POOLED, False),
    ("headline_r2", "headline_group_totals", R2, POOLED, True),
    ("headline_wape", "headline_group_totals", WAPE, POOLED, False),
    ("coverage", "coverage", "coverage", POOLED, True),
    ("jumpiness", "jumpiness", "mean_period_to_period_change", POOLED, False),
    ("scale_r2", "row_level_metadata_scale", R2, POOLED, True),
    ("scale_rmse", "row_level_metadata_scale", RMSE, POOLED, False),
    ("scale_within_10", "row_level_metadata_scale", WITHIN_TEN, POOLED, True),
    ("scale_within_25", "row_level_metadata_scale", WITHIN_TWENTY_FIVE, POOLED, True),
    ("hit_max_iter", "fit_diagnostics", "hit_max_iter", ALL_ROWS, False),
]
ABSOLUTE_METRICS = ["posts_abs_bias"]
FLAG_METRICS = ["hit_max_iter"]

## Setup

In [ ]:
import json
import re

import boto3
import polars as pl

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)

## Find the runs

Lists every file under each model's folder, then reads each run's small `metadata.json` for its time and
its scores. A run with a `metrics.parquet` is **cross_validated**; a run without one is **legacy** (scored
on a single 20% split of all rows with a target). The two are never compared with each other.

In [ ]:
s3 = boto3.client("s3")
keys = {
    obj["Key"]
    for model in MODEL_NAMES
    for page in s3.get_paginator("list_objects_v2").paginate(
        Bucket=BUCKET, Prefix=f"{MODELS_PREFIX}{model}/"
    )
    for obj in page.get("Contents", [])
}

run_rows = []
for key in sorted(keys):
    match = re.fullmatch(RUN_PATTERN + "metadata.json", key)
    if match:
        metadata = json.loads(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read())
        scores = metadata.get("metrics", {})
        has_metrics_file = key.replace("metadata.json", "metrics.parquet") in keys
        run_rows.append(
            {
                MODEL: match[1],
                VERSION: match[2],
                RUN: int(match[3]),
                TIMESTAMP: metadata[TIMESTAMP],
                EVALUATION: CROSS_VALIDATED if has_metrics_file else LEGACY,
                **{metric: scores.get(metric) for metric in METADATA_METRICS},
            }
        )

runs_df = (
    pl.DataFrame(run_rows, infer_schema_length=None)
    .with_columns(pl.col(TIMESTAMP).str.to_datetime(time_zone="UTC"))
    .sort(MODEL, VERSION, RUN)
)

## Runs per model and version

How many runs each model has, of which kind, and when the first and last ran. The run count and spacing
decide whether the rolling baseline has enough runs to work (`BASELINE_RUNS`).

In [ ]:
runs_df.group_by(MODEL, VERSION, EVALUATION).agg(
    pl.len().alias("runs"),
    pl.col(TIMESTAMP).min().alias("first_run"),
    pl.col(TIMESTAMP).max().alias("last_run"),
).sort(MODEL, VERSION, EVALUATION)

## Legacy scores

The four scores saved in `metadata.json` by legacy runs, one row per run. Read each model and version as its
own series; do not read them against the cross-validated scores below.

In [ ]:
runs_df.filter(pl.col(EVALUATION) == LEGACY).drop(EVALUATION)

## Load the cross-validated scores

Scans only the `metrics.parquet` files found above and keeps the pooled rows of `KEY_METRICS`, so a small
table is all that is collected: one row per run and tracked score. The version and run come from each file's
path.

In [ ]:
key_metrics_df = pl.DataFrame(
    KEY_METRICS,
    schema=[NAME, LEVEL, METRIC, FOLD, HIGHER_IS_BETTER],
    orient="row",
)
metrics_paths = [f"s3://{BUCKET}/{key}" for key in sorted(keys) if key.endswith("metrics.parquet")]

cv_metrics_df = (
    pl.scan_parquet(metrics_paths, include_file_paths="path", storage_options=STORAGE_OPTIONS)
    .filter(pl.col(GROUP) == ALL_GROUPS)
    .join(key_metrics_df.lazy(), on=[LEVEL, METRIC, FOLD])
    .select(
        pl.col(MODEL),
        pl.col("path").str.extract(RUN_PATTERN, 2).alias(VERSION),
        pl.col("path").str.extract(RUN_PATTERN, 3).cast(pl.Int64).alias(RUN),
        pl.col(NAME),
        pl.col(HIGHER_IS_BETTER),
        pl.when(pl.col(NAME).is_in(ABSOLUTE_METRICS))
        .then(pl.col(VALUE).abs())
        .otherwise(pl.col(VALUE))
        .alias(VALUE),
    )
    .collect()
)

## Cross-validated scores by run

One table per model: a row per run (oldest first within each version) and a column per tracked score.

In [ ]:
by_run_df = cv_metrics_df.join(runs_df.select(MODEL, VERSION, RUN, TIMESTAMP), on=[MODEL, VERSION, RUN])

for model_df in by_run_df.partition_by(MODEL, maintain_order=True):
    print(model_df[MODEL][0])
    print(model_df.pivot(on=NAME, index=[VERSION, RUN, TIMESTAMP], values=VALUE).sort(VERSION, RUN))

## Change against the previous run and the baseline

For each tracked score and run, within one version: `step_adverse` is the relative change against the
previous run and `drift_adverse` the relative change against the median of the previous `BASELINE_RUNS`
runs, both signed so that a positive number is a **worse** score. A breach is above `STEP_TOLERANCE` or
`DRIFT_TOLERANCE`. The first run of a version has no previous run, and the baseline needs `BASELINE_RUNS`
earlier runs, so those checks are empty until then. A previous value of exactly zero gives an infinite change.

In [ ]:
group = [MODEL, VERSION, NAME]
worse_when_higher = pl.when(pl.col(HIGHER_IS_BETTER)).then(-1.0).otherwise(1.0)

checks_df = (
    cv_metrics_df.filter(~pl.col(NAME).is_in(FLAG_METRICS))
    .sort(MODEL, VERSION, NAME, RUN)
    .with_columns(
        pl.col(VALUE).shift(1).over(group).alias("previous"),
        pl.col(VALUE)
        .shift(1)
        .rolling_median(BASELINE_RUNS, min_samples=BASELINE_RUNS)
        .over(group)
        .alias("baseline"),
    )
    .with_columns(
        (worse_when_higher * (pl.col(VALUE) - pl.col("previous")) / pl.col("previous").abs()).alias("step_adverse"),
        (worse_when_higher * (pl.col(VALUE) - pl.col("baseline")) / pl.col("baseline").abs()).alias("drift_adverse"),
    )
    .with_columns(
        (pl.col("step_adverse") > STEP_TOLERANCE).alias("step_breach"),
        (pl.col("drift_adverse") > DRIFT_TOLERANCE).alias("drift_breach"),
    )
)

### Breaches

Every run that would have raised an alert at the current tolerances.

In [ ]:
checks_df.filter(pl.col("step_breach") | pl.col("drift_breach")).select(
    MODEL, VERSION, RUN, NAME, VALUE, "previous", "baseline", "step_adverse", "drift_adverse"
).sort(MODEL, VERSION, RUN, NAME)

### Spread of changes

Per model, version and score: how many runs were compared, the typical (`median`) and worst (`max`) adverse
change, and how many breaches each tolerance gave. Choose tolerances above the usual run-to-run spread but
below the changes that matter.

In [ ]:
checks_df.group_by(MODEL, VERSION, NAME).agg(
    pl.col("step_adverse").count().alias("runs_compared"),
    pl.col("step_adverse").median().alias("median_step_adverse"),
    pl.col("step_adverse").max().alias("max_step_adverse"),
    pl.col("drift_adverse").max().alias("max_drift_adverse"),
    pl.col("step_breach").sum().alias("step_breaches"),
    pl.col("drift_breach").sum().alias("drift_breaches"),
).sort(MODEL, VERSION, NAME)

## Fits that hit the iteration limit

Runs where a `FLAG_METRICS` score is above zero. For `hit_max_iter`, those fits stopped at the iteration
limit before converging.

In [ ]:
cv_metrics_df.filter(pl.col(NAME).is_in(FLAG_METRICS) & (pl.col(VALUE) > 0)).sort(MODEL, VERSION, RUN)

## Charts: cross-validated scores

One figure per model, one panel per tracked score, a line per version, over the time each run was saved.
Needs `matplotlib` (`%pip install matplotlib` if the import fails).

In [ ]:
import matplotlib.pyplot as plt

chart_names = [name for name, *_ in KEY_METRICS if name not in FLAG_METRICS]
chart_rows = -(-len(chart_names) // CHART_COLUMNS)

for model in MODEL_NAMES:
    model_df = by_run_df.filter((pl.col(MODEL) == model) & ~pl.col(NAME).is_in(FLAG_METRICS))
    if model_df.is_empty():
        continue
    fig, axes = plt.subplots(chart_rows, CHART_COLUMNS, figsize=(5 * CHART_COLUMNS, 3 * chart_rows), squeeze=False)
    for ax, name in zip(axes.flat, chart_names):
        for (version,), version_df in model_df.filter(pl.col(NAME) == name).sort(RUN).group_by(VERSION, maintain_order=True):
            ax.plot(version_df[TIMESTAMP].to_list(), version_df[VALUE].to_list(), marker="o", label=version)
        ax.set_title(name, fontsize=9)
        ax.legend(fontsize=7)
    fig.suptitle(model)
    fig.tight_layout()
    plt.show()

## Charts: legacy scores

One row of panels per model, one panel per legacy score, a line per version. Kept separate from the
cross-validated charts because the scores are measured differently.

In [ ]:
legacy_df = runs_df.filter(pl.col(EVALUATION) == LEGACY)
fig, axes = plt.subplots(len(MODEL_NAMES), len(METADATA_METRICS), figsize=(5 * len(METADATA_METRICS), 3 * len(MODEL_NAMES)), squeeze=False)

for row, model in zip(axes, MODEL_NAMES):
    for ax, metric in zip(row, METADATA_METRICS):
        for (version,), version_df in legacy_df.filter(pl.col(MODEL) == model).group_by(VERSION, maintain_order=True):
            ax.plot(version_df[TIMESTAMP].to_list(), version_df[metric].to_list(), marker="o", label=version)
        ax.set_title(f"{model}: {metric}", fontsize=8)
        ax.legend(fontsize=7)

fig.tight_layout()
plt.show()